# Speech-to-Text Demo

Compares two Whisper checkpoints on your own voice, or on a shared bank of curated example clips:

- **`openai/whisper-large-v3`** — the original, multilingual Whisper.
- **[`kinit/whisper-large-v3-turbo-sk`](https://huggingface.co/kinit/whisper-large-v3-turbo-sk)** — Kinit's Whisper-large-v3-turbo, fine-tuned specifically for Slovak.

Structure:
1. Setup — imports
2. Load samples (common) — a shared, read-only bank of example clips, pulled from a public Drive folder
3. Create new samples (yours only) — record your own clips via the browser mic, kept for this session only
4. Load models
5. Transcribe — pick a sample + a model and compare

No personal Google Drive login or storage is used anywhere in this notebook — model weights and your own recordings just live on the Colab VM for the session and disappear when the runtime resets.

## 1. Setup: imports

In [ ]:
# Install missing packages. (gdown ships with Colab, but we pin -U for download_folder support.)
!pip install -q -U transformers accelerate pydub torchaudio gdown

In [ ]:
import os

# Common samples: a read-only bank of example clips curated ahead of time
# (interesting cases / failure modes) shared as a public Drive folder and
# downloaded fresh into this runtime in Section 2. No Drive mount or login
# needed - gdown fetches public folders directly.
COMMON_SAMPLES_DIR = "/content/common_samples"
COMMON_SAMPLES_FOLDER_ID = "1YgInfep4vnRA1pX4-7h8e4G4AxqzPLg6"

In [ ]:
import torch
from google.colab.output import eval_js
from IPython.display import Javascript, Audio, display
from base64 import b64decode
from pydub import AudioSegment
import io
import json
import numpy as np
import torchaudio

_RECORD_JS = """
async function record(ms, prompt) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

  const div = document.createElement('div');
  div.innerHTML = `<p>Click, then <b>${prompt}</b> (auto-stops after ${ms / 1000}s)</p><button>🎤 Start</button>`;
  document.body.appendChild(div);
  const btn = div.querySelector('button');
  await new Promise(resolve => { btn.onclick = resolve; });
  btn.textContent = `● Recording... (${ms / 1000}s)`;

  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  recorder.start();
  setTimeout(() => recorder.stop(), ms);
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  div.remove();

  const reader = new FileReader();
  reader.readAsDataURL(new Blob(chunks));
  return new Promise(resolve => { reader.onloadend = () => resolve(reader.result); });
}
"""

def _to_waveform(segment):
    segment = segment.set_channels(1)
    waveform = np.array(segment.get_array_of_samples()).astype(np.float32)
    waveform /= 1 << (8 * segment.sample_width - 1)
    return waveform, segment.frame_rate

def record_audio(seconds=8, prompt="speak now"):
    """Record from the browser mic, returns (waveform float32 numpy, sample_rate)."""
    display(Javascript(_RECORD_JS))
    data_url = eval_js(f"record({seconds * 1000}, {json.dumps(prompt)})")
    raw = b64decode(data_url.split(",", 1)[1])
    return _to_waveform(AudioSegment.from_file(io.BytesIO(raw)))

my_recordings = {}

def save_recording(name, waveform, sr):
    """Keep a recording in memory for this session (Section 3) - not written to disk or Drive, gone on runtime reset."""
    my_recordings[name] = (waveform, sr)

def load_sample(name):
    """Load a clip by name - checks your in-session recordings (Section 3) first, then the common bank (Section 2)."""
    if name in my_recordings:
        return my_recordings[name]
    path = f"{COMMON_SAMPLES_DIR}/{name}"
    if os.path.exists(path):
        return _to_waveform(AudioSegment.from_file(path))
    raise FileNotFoundError(f"'{name}' not found in your recordings or {COMMON_SAMPLES_DIR}")

def to_16k(waveform, sr):
    """Whisper expects 16kHz audio."""
    if sr == 16000:
        return waveform
    resampled = torchaudio.functional.resample(
        torch.from_numpy(waveform), orig_freq=sr, new_freq=16000
    )
    return resampled.numpy()

## 2. Load samples (common)

A small bank of example clips prepared ahead of time — interesting cases and failure modes (see the discussion at the end) — shared as a public Drive folder. Downloaded fresh into this runtime below, so there's no "add shortcut to your Drive" step and it doesn't touch your personal Drive quota.

**Risk:** if many attendees hit the same public folder at once, Google Drive's anti-abuse download quota can start throttling it — if this cell fails partway through the workshop, fall back to Section 3 and just record your own samples instead.

In [ ]:
os.makedirs(COMMON_SAMPLES_DIR, exist_ok=True)

try:
    import gdown
    gdown.download_folder(id=COMMON_SAMPLES_FOLDER_ID, output=COMMON_SAMPLES_DIR, quiet=False, use_cookies=False)
except Exception as e:
    print(f"Couldn't download the common sample bank ({e!r}) — you can still use Section 3 to record your own samples.")

common_samples = sorted(os.listdir(COMMON_SAMPLES_DIR))
print(f"{len(common_samples)} common sample(s): {common_samples}")

In [ ]:
# Preview the whole bank before picking one for Section 5.
for name in common_samples:
    print(name)
    display(Audio(f"{COMMON_SAMPLES_DIR}/{name}"))

## 3. Create new samples (yours only)

Record your own clips via the browser mic — allow microphone access when prompted, click **Start**, then say a sentence. Each recording is kept in memory for this session only (`my_recordings`, from Section 1) — nothing is written to disk or Drive, and it's gone once the runtime resets. Re-run a cell to retake; using the same name just overwrites it.

Use these names later in Section 5 by setting `sample_name`.

In [ ]:
waveform, sr = record_audio(6, prompt="say a sentence in English")
save_recording("english_example", waveform, sr)
display(Audio(waveform, rate=sr))

In [ ]:
waveform, sr = record_audio(6, prompt="say a sentence in Slovak")
save_recording("slovak_example", waveform, sr)
display(Audio(waveform, rate=sr))

## 4. Load models

Two ASR models to compare:

- **`openai/whisper-large-v3`** — the original, multilingual Whisper checkpoint (handles Slovak, English, and 90+ other languages, but Slovak is a small slice of its training data).
- **[`kinit/whisper-large-v3-turbo-sk`](https://huggingface.co/kinit/whisper-large-v3-turbo-sk)** — Kinit's Whisper-large-v3-**turbo** fine-tuned specifically on Slovak (Common Voice), reporting 9.29% WER vs. 29.23% for the un-tuned turbo model on the same Slovak test set. It's Slovak-only, though — fine-tuning on one language measurably degrades performance on everything else, so don't point it at English or other-language audio.

Both load into an `asr_pipelines` dict below, keyed by the names you'll pick from in Section 5.

In [ ]:
import torch
from transformers import pipeline

MODELS = {
    "whisper-large-v3": "openai/whisper-large-v3",
    "whisper-large-v3-turbo-sk": "kinit/whisper-large-v3-turbo-sk",
}

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
pipeline_device = 0 if device.startswith("cuda") else -1
print(device)

asr_pipelines = {
    name: pipeline(
        "automatic-speech-recognition",
        model=model_id,
        torch_dtype=dtype,
        device=pipeline_device,
    )
    for name, model_id in MODELS.items()
}

## 5. Transcribe: pick a sample + model

`transcribe(sample_name, model_name, language=None, task="transcribe")` ties everything together:

- `sample_name` — a filename from the common bank (Section 2's printed list, e.g. `"some_clip.wav"`) or a name you used with `save_recording()` in Section 3 (e.g. `"slovak_example"`, no extension — those live in memory, not as files).
- `model_name` — a key from `MODELS` (Section 4): `"whisper-large-v3"` or `"whisper-large-v3-turbo-sk"`.
- `language` — the spoken language of the sample (e.g. `"slovak"`, `"english"`). Required for the Slovak fine-tune; leave as `None` to let the original model auto-detect.
- `task` — `"transcribe"` (stays in the spoken language) or `"translate"` (always converts to English — only the original multilingual model supports this).

In [ ]:
def transcribe(sample_name, model_name, language=None, task="transcribe"):
    waveform, sr = load_sample(sample_name)
    display(Audio(waveform, rate=sr))

    generate_kwargs = {"task": task}
    if language:
        generate_kwargs["language"] = language

    result = asr_pipelines[model_name](
        {"array": to_16k(waveform, sr), "sampling_rate": 16000},
        generate_kwargs=generate_kwargs,
    )
    label = "Transcribed" if task == "transcribe" else "Translated to English"
    print(f"{label}: {result['text']}")
    return result["text"]

In [ ]:
# The Slovak fine-tune on your Slovak recording.
transcribe("slovak_example", "whisper-large-v3-turbo-sk", language="slovak")

In [ ]:
# The original multilingual model on your English recording.
transcribe("english_example", "whisper-large-v3", language="english")

### Translate too

Same mechanism, just flip `task` to `"translate"` — only the original multilingual model supports it (the Slovak fine-tune was trained purely for transcription):

In [ ]:
transcribe("slovak_example", "whisper-large-v3", language="slovak", task="translate")

## Common ASR failure modes

Even strong models struggle with — some of the common sample bank in Section 2 is chosen to showcase these live:

- **Named entities** — personal and place names, especially ones from a different language than the speech (e.g. Slovak surnames spoken in an English sentence, or vice versa).
- **Company/brand/product names** — anything newer, niche, or that sounds like an ordinary word, since the model has no domain knowledge to disambiguate from acoustics alone.
- **Numbers, dates, currency, units** — spoken-to-written conversion is inherently ambiguous ("twenty twenty-six" vs "2026"). This is exactly what the DER/DSER metrics in the `slovak-llm-audio` benchmark measure separately from word-level CER/WER.
- **Code-switching** — switching languages mid-sentence (e.g. a Slovak speaker dropping in English technical terms), since most models assume one language per utterance.
- **Low-resource languages** — bigger accuracy gaps for languages with less training data, like Slovak vs English — the reason a dedicated Slovak benchmark exists at all.
- **Homophones & rare words** — usually disambiguated by context, but that safety net is weaker for uncommon vocabulary.
- **Background noise, overlapping speech, multiple speakers** — crosstalk and noisy environments degrade accuracy sharply, and go beyond ASR into diarization.
- **Accents and dialects** — non-native or regional accents underrepresented in training data.
- **Hallucination on silence** — a well-documented Whisper-specific failure: near-silent audio can produce fabricated text instead of an empty transcript.
- **Disfluencies** — filler words, stutters, false starts; models may transcribe them literally, drop them inconsistently, or get confused by them.
- **Punctuation & capitalization** — there's no single "correct" punctuation for spoken language, so this is a common source of apparent errors that aren't really wrong words.